# Data

What this pipeline actually loads, and what it found when it did. Full provenance (sources, URLs, licences, per-vintage quirks) is in [`DATA.md`](../DATA.md) - this notebook computes real coverage numbers from the files themselves, live, rather than restating DATA.md in prose. See [`02_method.ipynb`](02_method.ipynb) for how these sources are combined into the counterfactual.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from council_tax_freeze.config import DATA_DIR, EXTENSION_FIRST_YEAR, HEADLINE_FIRST_YEAR, LAST_YEAR
from council_tax_freeze.boundaries.lad_2025 import LAD_2025_CODES
from council_tax_freeze.hpi.build import build_hpi
from council_tax_freeze.parsers.band_d.parse import build_band_d
from council_tax_freeze.parsers.ctsop.parse import build_ctsop
from council_tax_freeze.parsers.settlement.parse import build_settlement

BAND_D_FILE = DATA_DIR / "band_d" / "Band_D_1993_onwards.ods"
CTSOP_CONSOLIDATED = DATA_DIR / "ctsop" / "CTSOP1_0_1993_2024" / "CTSOP1_0_1993_2024_03_31.csv"
CTSOP_2025 = DATA_DIR / "ctsop" / "2025_summary.xlsx"
SETTLEMENT_FILE = DATA_DIR / "settlement" / "CSP_information_table_LGFS_2025-26.xlsx"
HPI_FILE = sorted((DATA_DIR / "hpi").glob("UK-HPI-full-file-*.csv"))[-1]

print(f"Study window: {EXTENSION_FIRST_YEAR} (backward extension start) to {LAST_YEAR}")
print(f"Headline window: {HEADLINE_FIRST_YEAR} to {LAST_YEAR}")
print(f"Target geography: {len(LAD_2025_CODES)} English LAs, 2025 vintage")

Study window: 2000-01 (backward extension start) to 2025-26
Headline window: 2009-10 to 2025-26
Target geography: 296 English LAs, 2025 vintage


## Band D council tax

MHCLG's maintained live table, "Band D council tax figures 1993-94 to 2026-27" - one continuously-updated file, not 26 separate annual releases. Covers every predecessor-level identity back to 1993-94.

In [2]:
bd = build_band_d(BAND_D_FILE)
print(f"{len(bd.la_year):,} (LA, year) rows, {bd.la_year['financial_year'].nunique()} distinct financial years")
print(f"National validation: {bd.validation}")
bd.coverage.tail(10)

12,648 (LA, year) rows, 34 distinct financial years
National validation:    financial_year  live_table_value  independent_anchor within_tolerance
0         1993-94            567.95                 NaN             None
1         1994-95            580.11                 NaN             None
2         1995-96            609.00                 NaN             None
3         1996-97            646.00                 NaN             None
4         1997-98            688.00                 NaN             None
5         1998-99            747.00                 NaN             None
6         1999-00            798.00                 NaN             None
7         2000-01            847.00              847.00             True
8         2001-02            901.33              901.00             True
9         2002-03            975.56              976.00             True
10        2003-04           1101.75             1102.00             True
11        2004-05           1166.56             116

,financial_year,n_la_rows,n_unmapped,n_ambiguous
24,2017-18,326,0,0
25,2018-19,326,0,0
26,2019-20,317,0,0
27,2020-21,314,0,0
28,2021-22,309,0,0
29,2022-23,309,0,0
30,2023-24,296,0,0
31,2024-25,296,0,0
32,2025-26,296,0,0
33,2026-27,296,0,0


## VOA CTSOP: Council Tax Stock of Properties (band distribution)

Two files: a VOA-maintained consolidated "1993 to 2024" time series, plus the standalone 2025 release for the one year not yet folded in.

**A real data-quality finding, not a parsing bug:** the consolidated file double-counts every LA affected by the 2019/2020/2021/2023 reorg waves - VOA correctly back-filled the successor row with the retroactive predecessor sum, but then left the predecessor rows in place, still updating with real dwelling counts for years after their own legal abolition. Fixed in `parsers/ctsop/parse.py` by dropping predecessor rows for MERGE events; see `duplication_check` below and DATA.md for the full account (Dorset UA's 2015 total verified to equal the exact sum of its five predecessors, and East Dorset shown still growing 5 years after its 2019 abolition - the smoking gun).

In [3]:
ct = build_ctsop(CTSOP_CONSOLIDATED, CTSOP_2025)
print(f"{len(ct.la_year):,} de-duplicated (LA, year) rows")
print(f"Predecessor weights preserved (not discarded): {len(ct.predecessor_weights):,} rows")
print()
print("Duplication check (national dwelling total, before vs after the fix):")
ct.duplication_check

9,472 de-duplicated (LA, year) rows
Predecessor weights preserved (not discarded): 1,116 rows

Duplication check (national dwelling total, before vs after the fix):


,financial_year,raw_la_sum,natl_row,excess
0,1994-95,21603040,20200200,1402840
1,1995-96,21785270,20367140,1418130
2,1996-97,21981120,20547920,1433200
3,1997-98,22125950,20707790,1418160
4,1998-99,22295470,20863090,1432380
5,1999-00,22463240,21017600,1445640
6,2000-01,22625570,21167190,1458380
7,2001-02,22770120,21299080,1471040
8,2002-03,22922080,21437010,1485070
9,2003-04,23081050,21583230,1497820


## UK House Price Index

LA-level revaluation factors, anchored to January 1995 (the earliest month LA-level HPI data exists - not April 1991, the true valuation date; that four-year gap is real and deliberately unbridged, see README "Framing"). Validated independently against IFS (2020)'s own published regional ratios - not just checked for internal consistency, checked against an external, already-published number.

In [4]:
hpi = build_hpi(HPI_FILE)
print(f"{len(hpi.la_factors):,} (LA, year) HPI factor rows")
print()
print("Coverage (LAs with a real vs regional-fallback series), most recent 5 years:")
display(hpi.coverage.tail(5))
print()
print("Validation against IFS (2020)'s own published regional ratios:")
hpi.validation

7,696 (LA, year) HPI factor rows

Coverage (LAs with a real vs regional-fallback series), most recent 5 years:


,financial_year,n_la,n_fallback,n_missing
21,2021-22,296,1,0
22,2022-23,296,1,0
23,2023-24,296,1,0
24,2024-25,296,1,0
25,2025-26,296,1,0



Validation against IFS (2020)'s own published regional ratios:


,region_code,name,ratio,anchor_min,anchor_max,within_tolerance
0,E12000007,London,6.30,6.0,inf,True
1,E12000001,North East,3.08,2.9,3.3,True


## MHCLG Core Spending Power (settlement data, Phase 6)

One maintained file, one sheet per year back to 2015-16 - "Core Spending Power" did not exist as a concept before then (earlier years used Formula Grant, then early Settlement Funding Assessment), so this is out of scope by design for years before 2015-16. Four row tiers per sheet (district/unitary/borough, county, Greater London Authority, Greater Manchester Combined Authority) - the upper tiers are apportioned to their constituent 2025-vintage LAs by dwelling share; see `parsers/settlement/parse.py` module docstring for exactly how, and the one known, quantified gap (5 counties that unitarised mid-window).

In [5]:
settlement = build_settlement(SETTLEMENT_FILE, ct.la_year)
print(f"{len(settlement.la_year):,} (LA, year) rows, {settlement.la_year['financial_year'].nunique()} years (2015-16 to 2025-26)")
print(f"Unresolved rows: {len(settlement.unresolved)}")
print()
print("Coverage and the known transitional-county apportionment gap, by year:")
settlement.coverage

3,256 (LA, year) rows, 11 years (2015-16 to 2025-26)
Unresolved rows: 0

Coverage and the known transitional-county apportionment gap, by year:


,financial_year,n_la,n_unapportioned_upper_tier_gap
0,2015-16,296,31
1,2016-17,296,31
2,2017-18,296,31
3,2018-19,296,31
4,2019-20,296,24
5,2020-21,296,24
6,2021-22,296,17
7,2022-23,296,17
8,2023-24,296,0
9,2024-25,296,0


## Boundary harmonisation

Every LA identity active at any point since 2000-01 resolves to exactly one 2025-vintage LA (or a weighted split, for the one genuine SPLIT event - Barnsley/Sheffield's 2025 Oughtibridge Mill transfer). 22 structural changes (RECODE/MERGE/SPLIT) across five reorg waves (2009, 2019, 2020, 2021, 2023) plus the 2025 boundary tweak, each cited against the ONS Code History Database or an equivalent maintained reference - see `boundaries/reorg_events.py` and DATA.md "Boundary harmonisation" for the full list and citations. `tests/test_boundaries.py::test_full_2000_2025_coverage` checks this holds for every LA identity in every financial year, not just a handful of examples.

## What these sources do and don't agree on

Two independent cross-validations worth surfacing here, not just in DATA.md prose:

- **Table 125's Old/New ONS code columns** (a separate MHCLG series, dwelling stock estimates) independently corroborate the Phase 1 boundary crosswalk built from the ONS Code History Database - two different sources agreeing on the same set of LA identity changes.
- **The UK HPI file's regional ratios reproduce IFS (2020)'s own published numbers** (London >6x January 1995 level by November 2019; North East "barely three times") directly, before this pipeline's own counterfactual engine even exists - the first point of contact between this pipeline's inputs and IFS's own published results, and it lands.

## Reproducing this notebook

```
make data   # fetches everything download.py can fetch automatically; prints instructions for the rest
jupyter nbconvert --to notebook --execute notebooks/01_data.ipynb
```